# Cohere model matrix: measured (Colab GPU)

Turns `docs/model_backends.md`'s table from a registry reading into a measurement.
Two questions this answers that reading source cannot:

1. **Which models actually answer a text prompt**, scored on the *answer* rather
   than on whether a string came back -- a multilingual model can load cleanly and
   still reply in English.
2. **Does a vision architecture work with its vision tower unused?** Asked in the
   review, and answered here: yes for `aya_vision` on `hf:`.

| Model | arch | needs |
|---|---|---|
| Tiny Aya Global / Fire / Water / Earth | `cohere2` | ~8 GB |
| North Micro Vision Instruct | `cohere_compass` | ~6 GB |
| Aya Expanse 8B | `cohere` | ~18 GB |
| Aya Vision 8B | `aya_vision` | ~20 GB |
| Aya Expanse 32B | `cohere` | ~70 GB |
| Aya Vision 32B | `aya_vision` | ~75 GB |

**Use `--gpu G4`** (RTX PRO 6000, 95 GB) to cover all nine. `L4` (22 GB) covers the
first four rows; `T4` (15 GB) only the first two.

**Runtime:** Colab. All outputs are cleared; the table at the end is the result.

In [ ]:
# ---- settings ----
BRANCH = "main"          # which branch of AuditKIT to test
PORT = 30000
# (repo, architecture, min GPU GB in bf16, is_vision_arch)
MODELS = [
    ("CohereLabs/tiny-aya-global", "cohere2", 8,  False),
    ("CohereLabs/tiny-aya-fire",   "cohere2", 8,  False),
    ("CohereLabs/tiny-aya-water",  "cohere2", 8,  False),
    ("CohereLabs/tiny-aya-earth",  "cohere2", 8,  False),
    ("CohereLabs/North-Micro-Vision-Instruct", "cohere_compass", 6, True),
    ("CohereLabs/aya-expanse-8b",  "cohere", 18, False),
    ("CohereLabs/aya-vision-8b",   "aya_vision", 20, True),
    ("CohereLabs/aya-expanse-32b", "cohere", 70, False),
    ("CohereLabs/aya-vision-32b",  "aya_vision", 75, True),
]

# `expect` is any one acceptable answer. Asking is not enough: a model that loads
# and answers in English scores the same as one that understood the question.
PROBES = [
    ("fr", "Quel est le plus grand port de France ? Un mot.",
     ("Marseille", "marseille", "Marseill")),
    ("hi", "\u092d\u093e\u0930\u0924 \u0915\u0940 \u0930\u093e\u091c\u0927\u093e\u0928\u0940 \u0915\u094d\u092f\u093e \u0939\u0948? \u090f\u0915 \u0936\u092c\u094d\u0926\u0964",
     ("\u0926\u093f\u0932\u094d\u0932\u0940", "\u0928\u090f\u0940 \u0926\u093f\u0932\u094d\u0932\u0940", "delhi")),
    ("zh", "\u4e2d\u56fd\u7684\u9996\u90fd\u662f\u54ea\u91cc\uff1f\u7528\u4e00\u4e2a\u8bcd\u3002",
     ("\u5317\u4eac", "beijing")),
    ("ar", "\u0645\u0627 \u0647\u064a \u0639\u0627\u0635\u0645\u0629 \u0645\u0635\u0631\uff1f \u0643\u0644\u0645\u0629 \u0648\u0627\u062d\u062f\u0629.",
     ("\u0627\u0644\u0642\u0627\u0647\u0631\u0629", "cairo")),
    ("th", "\u0e40\u0e21\u0e37\u0e2d\u0e07\u0e2b\u0e25\u0e27\u0e07\u0e02\u0e2d\u0e07\u0e1b\u0e23\u0e30\u0e40\u0e17\u0e28\u0e44\u0e17\u0e22\u0e04\u0e37\u0e2d\u0e2d\u0e30\u0e32\u0e2b\u0e48\u0e2d\u0e04\u0e23\u0e31\u0e1a? \u0e2b\u0e19\u0e34\u0e48\u0e07\u0e04\u0e33",
     ("\u0e01\u0e23\u0e38\u0e07\u0e40\u0e17\u0e1e", "bangkok")),
]
RESULTS = []

## 1 · Install

**Install from this repository, and check the installed version.** The notebook used to install from
`Lexsi-Labs/AuditKIT-Internal`, a separate, stale private repository still on the 1.0.0 extras
(`transformers>=4.44,<5`). pip obeyed that cap, silently downgrading transformers to 4.57.6, which
has no `cohere_compass`, so North Micro Vision failed with an error that read like a model problem
(#51). This repository's `[transformers]` extra is `transformers>=5.15,<6`, so no re-pinning is needed.

The check below stops the notebook unless the installed `auditkit` is 1.2 or newer. That catches a
wrong repository, which a transformers-version check can't tell apart from a legitimate constraint.
It also prints the requirements that actually applied.

In [ ]:
# ---- tokens: paste into the box that appears (hidden input; never saved in the notebook) ----
import os
from getpass import getpass
for key, why in (("GITHUB_TOKEN", "required: installs AuditKIT from the private repo"),
                 ("HF_TOKEN", "required: 8 of the 9 models are gated on the Hub")):
    if os.environ.get(key):
        print(f"{key}: already set in this runtime"); continue
    value = getpass(f"Paste {key} ({why}): ").strip()
    if value: os.environ[key] = value
print({k: bool(os.environ.get(k)) for k in ("GITHUB_TOKEN", "HF_TOKEN")})

In [ ]:
import subprocess, sys
from importlib import metadata
def sh(*a): return subprocess.run(a, capture_output=True, text=True)
def tfver(): return sh(sys.executable, "-c", "import transformers;print(transformers.__version__)").stdout.strip()

print("preinstalled transformers:", tfver() or "IMPORT FAILED")
# this repository -- NOT Lexsi-Labs/AuditKIT-Internal, a stale separate repo on the 1.0.0 extras (#51)
url = f"https://{os.environ['GITHUB_TOKEN']}@github.com/aryapratinavseth/AuditKIT-Internal.git"
r = sh(sys.executable, "-m", "pip", "install", "-q",
       f"auditkit[transformers,vision] @ git+{url}@{BRANCH}")
print(f"auditkit install rc={r.returncode}" + (f"\n{r.stderr[-800:]}" if r.returncode else ""))

# the version alone catches a wrong repository; print the constraints that actually applied
installed = sh(sys.executable, "-c", "import importlib.metadata as m; print(m.version('auditkit'))").stdout.strip()
requires = sh(sys.executable, "-c", "import importlib.metadata as m; print(m.requires('auditkit'))").stdout.strip()
print("installed auditkit:", installed)
print("its requirements:", requires)
if tuple(int(x) for x in (installed or "0.0").split(".")[:2]) < (1, 2):
    raise SystemExit(f"auditkit {installed or 'missing'} is not this repository's code (1.2+): check REPO/BRANCH. See #51.")
got = tfver()
print("transformers:", got)
if not got.startswith("5."):
    raise SystemExit(f"transformers {got}: need 5.x (>=5.15) for cohere_compass. See the requirements above.")
import torch, transformers
from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
print(f"torch {torch.__version__} | GPU {torch.cuda.get_device_name(0)} "
      f"{torch.cuda.get_device_properties(0).total_memory/2**30:.0f} GB")
print("registered:", {m: (m in CONFIG_MAPPING_NAMES)
                      for m in ("cohere", "cohere2", "aya_vision", "cohere_compass")})

## 2 · The sweep

One model at a time, five languages, scored on the answer. Read the table, not the
exit code -- `SKIP` and `FAIL` are both findings.

In [ ]:
import gc, time, traceback
import torch, transformers
import auditkit as ak
from auditkit.model import Request
from auditkit.model.hf_gen import HFGenModel

GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0

def record(stage, test, outcome, detail=""):
    RESULTS.append({"stage": stage, "test": test, "outcome": outcome, "detail": str(detail)[-500:]})
    print(f"[{outcome:8}] {stage} / {test}")
    if detail: print("           " + str(detail)[:400], flush=True)

for repo, arch, need, is_vision in MODELS:
    stage = f"hf:{repo}"
    if GPU_GB < need:
        record(stage, "load", "SKIP", f"{arch} needs ~{need} GB in bf16, GPU has {GPU_GB:.0f}"); continue
    if not os.environ.get("HF_TOKEN"):
        record(stage, "load", "SKIP", "gated and no HF_TOKEN"); continue
    t0, model = time.time(), None
    try:
        model = HFGenModel(repo, device="cuda", name=f"hf:{repo}")
        langs, right, wrong = {}, 0, 0
        for code, prompt, expect in PROBES:
            try:
                # No image anywhere: the point is the TEXT path, including on a vision arch.
                out = model.generate([Request(prompt=prompt, params={"messages": [
                    {"role": "user", "content": prompt}]})])[0].completions[0].text
                text = (out or "").strip().replace("\n", " ")[:70]
                langs[code] = text
                if any(e in text for e in expect): right += 1
                else: wrong += 1
            except Exception as e:
                langs[code] = f"ERR {type(e).__name__}"; wrong += 1
        record(stage, "text-generate",
               "PASS" if right >= 4 else ("PARTIAL" if right else "FAIL"),
               f"{arch}{' VISION-ARCH' if is_vision else ''} {right}/{len(PROBES)} correct, "
               f"{time.time()-t0:.0f}s :: " + " | ".join(f"{k}={v!r}" for k, v in langs.items()))
    except Exception as e:
        record(stage, "load", "FAIL", f"{type(e).__name__}: {e}")
        traceback.print_exc()
    finally:
        del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
import json
print("=" * 100)
for r in RESULTS:
    print(f"{r['outcome']:8} {r['stage']:46} {r['test']}")
print("=" * 100)
counts = {}
for r in RESULTS: counts[r["outcome"]] = counts.get(r["outcome"], 0) + 1
print("TOTALS:", counts, "| GPU", f"{GPU_GB:.0f} GB", "| transformers", transformers.__version__)
with open("cohere_matrix_results.json", "w") as f:
    json.dump({"gpu_gb": round(GPU_GB, 1), "transformers": transformers.__version__,
               "results": RESULTS}, f, indent=1)
print("wrote cohere_matrix_results.json")